In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Make sure you have tf-keras installed
# !pip install -U tf-keras transformers

import tf_keras as keras
from tf_keras import layers, models, optimizers, metrics
from transformers import TFAutoModel, AutoTokenizer

# Load base model
base_model = TFAutoModel.from_pretrained("bert-base-uncased")
base_model.trainable = True

# Inputs
max_len = 200
input_ids = layers.Input(shape=(max_len,), dtype="int32", name="input_ids")
attention_mask = layers.Input(shape=(max_len,), dtype="int32", name="attention_mask")

# Forward pass
outputs = base_model(input_ids=input_ids, attention_mask=attention_mask)
last_hidden = outputs.last_hidden_state
cls = layers.Lambda(lambda x: x[:, 0, :])(last_hidden)

# Classifier
x = layers.Dropout(0.3)(cls)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
logit = layers.Dense(1, activation="sigmoid")(x)

# Build model
model = models.Model([input_ids, attention_mask], logit)

# Compile (note: use tf_keras.optimizers)
model.compile(
    optimizer=optimizers.AdamW(learning_rate=2e-5, weight_decay=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy", metrics.AUC(name="AUC")]
)

model.summary()


In [ ]:
df_train=pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df_test=pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")
encoding_train=tokenizer(list(df_train['text'].values),padding='max_length',max_length=200,truncation=True)
encoding_test=tokenizer(list(df_test['text'].values),padding='max_length',max_length=200,truncation=True)
y=df_train['target'].values
from sklearn.model_selection import train_test_split
X_input_train,X_input_val,X_attention_train,X_attention_val,y_train,y_val=train_test_split(np.array(encoding_train['input_ids']),np.array(encoding_train['attention_mask']),y,test_size=0.2,stratify=y)

In [ ]:
model.fit([X_input_train,X_attention_train],y_train,validation_data=([X_input_val,X_attention_val],y_val),epochs=3)

In [ ]:
X_input_test,X_attention_test=np.array(encoding_test['input_ids']),np.array(encoding_test['attention_mask'])

In [ ]:
y_test_proba=model.predict([X_input_test,X_attention_test])
y_test=(y_test_proba>0.5).astype(int)
y_test=y_test.flatten()

In [ ]:
res=pd.DataFrame({'id':df_test['id'],'target':y_test})
res.to_csv("fine_tuned3.csv",index=False)